# Práctica 3: Modelos Multimodales con Phi-4-multimodal-instruct

Este notebook demuestra el uso de modelos multimodales en Azure Foundry con:
1. **Chat con imágenes** - Análisis y descripción de imágenes desde URLs y archivos locales
2. **Chat con audio** - Procesamiento y traducción de archivos de audio
3. **Prompts multimodales** - Combinación de imagen + texto, audio + texto
4. **Control de formatos** - Manejo de diferentes tipos de contenido

## Requisitos previos:
- Deployment: `Phi-4-multimodal-instruct` en Azure Foundry
- API Endpoint y Key configurados en `.env`
- Paquete: `azure-ai-inference`
- Conexión a internet (para descargar imágenes de prueba)

## 1. Configuración e Instalación

In [20]:
# Instalar dependencias necesarias
import subprocess
import sys

# Azure AI Inference es el SDK correcto para Foundry (no OpenAI SDK)
packages = ['azure-ai-inference>=1.0.0', 'azure-identity>=1.15.0', 'pillow>=10.0.0', 'python-dotenv>=1.0.0', 'requests>=2.31.0']
for package in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    except:
        print(f"⚠️  Error instalando {package}")

print('✓ Dependencias instaladas correctamente')
print('  - azure-ai-inference (for Foundry)')
print('  - azure-identity')
print('  - pillow (image processing)')
print('  - python-dotenv (environment variables)')
print('  - requests (http requests)')

⚠️  Error instalando azure-ai-inference>=1.0.0
✓ Dependencias instaladas correctamente
  - azure-ai-inference (for Foundry)
  - azure-identity
  - pillow (image processing)
  - python-dotenv (environment variables)
  - requests (http requests)


In [21]:
# Cargar variables de entorno (force reload)
from dotenv import load_dotenv
import os

# Limpiar cache anterior
if 'DEPLOYMENT_NAME' in os.environ:
    del os.environ['DEPLOYMENT_NAME']
if 'FOUNDRY_ENDPOINT' in os.environ:
    del os.environ['FOUNDRY_ENDPOINT']
if 'FOUNDRY_API_KEY' in os.environ:
    del os.environ['FOUNDRY_API_KEY']

# Cargar desde .env.notebook3 (force reload=True)
load_dotenv(dotenv_path='.env.notebook3', override=True)

# Configurar variables para Foundry
FOUNDRY_ENDPOINT = os.getenv("FOUNDRY_ENDPOINT", "")
FOUNDRY_API_KEY = os.getenv("FOUNDRY_API_KEY", "")
DEPLOYMENT_NAME = os.getenv("DEPLOYMENT_NAME", "")

# Para azure.ai.inference, el endpoint debe terminar en /models
# Ejemplo: https://foundryjnb.services.ai.azure.com/models
if FOUNDRY_ENDPOINT and "/api/projects/" in FOUNDRY_ENDPOINT:
    base = FOUNDRY_ENDPOINT.split("/api/projects/")[0]
    ENDPOINT_FOR_SDK = f"{base}/models"
else:
    ENDPOINT_FOR_SDK = FOUNDRY_ENDPOINT

# Validar configuración
if not FOUNDRY_ENDPOINT or not FOUNDRY_API_KEY or not DEPLOYMENT_NAME:
    print("⚠️  CONFIGURACIÓN NECESARIA:")
    print("\n   Verifica que .env.notebook3 tenga:")
    print("   FOUNDRY_ENDPOINT=https://foundryjnb.services.ai.azure.com/api/projects/proj-default")
    print("   FOUNDRY_API_KEY=<your-api-key>")
    print("   DEPLOYMENT_NAME=Phi-4-multimodal-instruct")
else:
    print(f"✓ Variables de entorno cargadas correctamente")
    print(f"  Endpoint para SDK: {ENDPOINT_FOR_SDK[:60]}...")
    print(f"  Deployment: {DEPLOYMENT_NAME}")

✓ Variables de entorno cargadas correctamente
  Endpoint para SDK: https://foundryjnb.services.ai.azure.com/models...
  Deployment: Phi-4-multimodal-instruct


In [22]:
# Imports for multimodal AI with Foundry
import base64
import json
from pathlib import Path
import requests
from io import BytesIO
from PIL import Image, ImageDraw

# Import azure.ai.inference - Official SDK for Foundry
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    AudioContentItem,
    InputAudio,
    AudioContentFormat,
    SystemMessage,
    UserMessage
)
from azure.core.credentials import AzureKeyCredential

print("✓ Imports completados correctamente")
print("  - azure.ai.inference.ChatCompletionsClient")
print("  - azure.ai.inference.models (TextContentItem, ImageContentItem, AudioContentItem)")
print("  - PIL (image processing)")

✓ Imports completados correctamente
  - azure.ai.inference.ChatCompletionsClient
  - azure.ai.inference.models (TextContentItem, ImageContentItem, AudioContentItem)
  - PIL (image processing)


In [23]:
# Crear cliente de Foundry con azure.ai.inference
print("═" * 70)
print("CLIENTE FOUNDRY: Inicializando ChatCompletionsClient")
print("═" * 70)

try:
    client = ChatCompletionsClient(
        endpoint=ENDPOINT_FOR_SDK,
        credential=AzureKeyCredential(FOUNDRY_API_KEY),
        model=DEPLOYMENT_NAME
    )
    print("✓ Cliente Foundry creado correctamente")
    print(f"  Endpoint: {ENDPOINT_FOR_SDK}")
    print(f"  Deployment: {DEPLOYMENT_NAME}")
    print(f"  Authentication: AzureKeyCredential")
    print("\n✓ Listo para procesar contenido multimodal (texto, imágenes, audio)")
except Exception as e:
    print(f"✗ Error creando cliente: {e}")
    client = None

print("═" * 70)

══════════════════════════════════════════════════════════════════════
CLIENTE FOUNDRY: Inicializando ChatCompletionsClient
══════════════════════════════════════════════════════════════════════
✓ Cliente Foundry creado correctamente
  Endpoint: https://foundryjnb.services.ai.azure.com/models
  Deployment: Phi-4-multimodal-instruct
  Authentication: AzureKeyCredential

✓ Listo para procesar contenido multimodal (texto, imágenes, audio)
══════════════════════════════════════════════════════════════════════


In [24]:
# Test1: Simple text prompt to verify API connectivity
print("═" * 70)
print("TEST 1: Verificar conectividad con Foundry")
print("═" * 70)

try:
    response = client.complete(
        messages=[
            SystemMessage("You are a helpful assistant."),
            UserMessage(content=[
                TextContentItem(text="¿Cuál es 2 + 2?")
            ]),
        ],
        temperature=0.7,
        max_tokens=100,
    )
    
    print("✓ ¡Conexión exitosa con Foundry!")
    print(f"\nRespuesta: {response.choices[0].message.content}")
    print(f"\nDetalles:")
    print(f"  Modelo: {response.model}")
    print(f"  Prompt tokens: {response.usage.prompt_tokens}")
    print(f"  Completion tokens: {response.usage.completion_tokens}")
    
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

print("═" * 70)

══════════════════════════════════════════════════════════════════════
TEST 1: Verificar conectividad con Foundry
══════════════════════════════════════════════════════════════════════
✓ ¡Conexión exitosa con Foundry!

Respuesta: 2 + 2 es 4. Es una suma aritmética básica.

Detalles:
  Modelo: Phi-4-multilmodal-instruct
  Prompt tokens: 20
  Completion tokens: 17
══════════════════════════════════════════════════════════════════════


In [25]:
# Verificación de configuración de Foundry
print("\n" + "═" * 70)
print("DIAGNÓSTICO: Verificación de Configuración de Foundry")
print("═" * 70)

print(f"\n📋 Variables configuradas en memoria:")
print(f"  FOUNDRY_ENDPOINT: {FOUNDRY_ENDPOINT[:50] if FOUNDRY_ENDPOINT else 'NO CONFIGURADO'}...")
print(f"  ENDPOINT_FOR_SDK: {ENDPOINT_FOR_SDK[:50] if ENDPOINT_FOR_SDK else 'NO CONFIGURADO'}...")
print(f"  DEPLOYMENT_NAME: {DEPLOYMENT_NAME}")
print(f"  FOUNDRY_API_KEY: {FOUNDRY_API_KEY[:20] if FOUNDRY_API_KEY else 'NO CONFIGURADO'}...***")
print(f"  Cliente tipo: {type(client).__name__ if client else 'NO INICIALIZADO'}")

print(f"\n✓ Verificación completada")
print(f"  All necessary variables are defined and client is ready")

print("\n" + "═" * 70)


══════════════════════════════════════════════════════════════════════
DIAGNÓSTICO: Verificación de Configuración de Foundry
══════════════════════════════════════════════════════════════════════

📋 Variables configuradas en memoria:
  FOUNDRY_ENDPOINT: https://foundryjnb.services.ai.azure.com/api/proje...
  ENDPOINT_FOR_SDK: https://foundryjnb.services.ai.azure.com/models...
  DEPLOYMENT_NAME: Phi-4-multimodal-instruct
  FOUNDRY_API_KEY: EGTx4ao0mDEGjLUPjbjU...***
  Cliente tipo: ChatCompletionsClient

✓ Verificación completada
  All necessary variables are defined and client is ready

══════════════════════════════════════════════════════════════════════


## 2. Part 1: Análisis Básico de Imágenes

El modelo `Phi-4-multimodal-instruct` puede analizar imágenes desde URLs públicas y archivos locales.

In [26]:
## Image Analysis Functions using azure.ai.inference

def analyze_image_from_url(image_url: str, prompt: str):
    """Analiza una imagen desde URL usando Phi-4-multimodal-instruct.
    
    Args:
        image_url: URL pública de la imagen
        prompt: Pregunta/análisis a realizar
    
    Returns:
        Response object con análisis del modelo
    
    Raises:
        Exception: Si falla la petición a la API
    """
    if not client:
        raise ValueError("Cliente no configurado")
    
    # Usar el formato correcto de azure.ai.inference
    data_url = ImageUrl(url=image_url)
    
    response = client.complete(
        messages=[
            SystemMessage("You are a helpful assistant that can analyze images and provide detailed descriptions."),
            UserMessage(content=[
                TextContentItem(text=prompt),
                ImageContentItem(image_url=data_url)
            ]),
        ],
        temperature=1,
        max_tokens=2048,
    )
    
    return response


def analyze_image_from_base64(image_path: str, prompt: str):
    """Analiza una imagen local (convertida a base64).
    
    Args:
        image_path: Ruta local del archivo
        prompt: Análisis a realizar
    
    Returns:
        Response object con análisis del modelo
    
    Raises:
        FileNotFoundError: Si el archivo no existe
        Exception: Si falla la petición a la API
    """
    if not client:
        raise ValueError("Cliente no configurado")
    
    # Leer y codificar imagen
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Archivo no encontrado: {image_path}")
    
    with open(image_path, 'rb') as img_file:
        image_data = base64.b64encode(img_file.read()).decode('utf-8')
    
    # Determinar media type
    path_lower = str(image_path).lower()
    if path_lower.endswith('.png'):
        media_type = "image/png"
    elif path_lower.endswith('.gif'):
        media_type = "image/gif"
    elif path_lower.endswith('.webp'):
        media_type = "image/webp"
    else:
        media_type = "image/jpeg"
    
    # Crear data URL
    data_url = ImageUrl(url=f"data:{media_type};base64,{image_data}")
    
    response = client.complete(
        messages=[
            SystemMessage("You are a helpful assistant that can analyze images and provide detailed descriptions."),
            UserMessage(content=[
                TextContentItem(text=prompt),
                ImageContentItem(image_url=data_url)
            ]),
        ],
        temperature=1,
        max_tokens=2048,
    )
    
    return response

print("✓ Funciones de análisis de imágenes definidas (azure.ai.inference SDK)")

✓ Funciones de análisis de imágenes definidas (azure.ai.inference SDK)


In [27]:
# TEST 2: Análisis de imagen local - stardew-valley.webp
print("\n" + "═" * 70)
print("TEST 2: Análisis de Imagen Local (stardew-valley.webp)")
print("═" * 70)

image_path = "stardew-valley.webp"
prompt = "Describe this image in detail. What do you see? What are the main elements and colors?"

print(f"\nArchivo: {image_path}")
print(f"Prompt: {prompt}")
print("\nPreparada para análisis...\n")

try:
    result = analyze_image_from_base64(image_path, prompt)
    analysis = result.choices[0].message.content
    
    print(f"✓ Análisis exitoso:")
    print(f"\n{analysis}")
    print(f"\nDetalles:")
    print(f"  Tokens utilizados: {result.usage.total_tokens}")
    print(f"  Modelo: {result.model}")
    
except FileNotFoundError as e:
    print(f"✗ Archivo no encontrado: {e}")
    raise
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    raise


══════════════════════════════════════════════════════════════════════
TEST 2: Análisis de Imagen Local (stardew-valley.webp)
══════════════════════════════════════════════════════════════════════

Archivo: stardew-valley.webp
Prompt: Describe this image in detail. What do you see? What are the main elements and colors?

Preparada para análisis...

✓ Análisis exitoso:

The image is a screenshot of a pixel art scene, likely from a game. It features a wooden house with a red roof and window, located in a grassy area with scattered rocks and trees. The game menu indicates the time as 8:30 am and health points of 140/140. A character with brown hair and blue clothes stands near a sign that says "Welcome." Various objects such as colored blocks, arrows, and items are scattered around the scene.

Detalles:
  Tokens utilizados: 1730
  Modelo: vision


## 3. Part 2: Análisis Avanzado de Imágenes

In [28]:
# TEST 4: Extracción de texto (OCR)
print("\n" + "═" * 70)
print("TEST 4: Extracción de Texto (OCR)")
print("═" * 70)

image_path_ocr = "2xpmanzdwefa1.jpg"
prompt_ocr = "Extract and read all text visible in this image. List it exactly as written."

print(f"\nArchivo: {image_path_ocr}")
print(f"Prompt: {prompt_ocr}")
print("\nExtrayendo texto...\n")

try:
    result = analyze_image_from_base64(image_path_ocr, prompt_ocr)
    analysis = result.choices[0].message.content
    
    print(f"✓ Texto extraído:")
    print(f"\n{analysis}")
    print(f"\nDetalles:")
    print(f"  Tokens utilizados: {result.usage.total_tokens}")
    print(f"  Modelo: {result.model}")
    
except FileNotFoundError as e:
    print(f"✗ Archivo no encontrado: {e}")
    raise
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    raise


══════════════════════════════════════════════════════════════════════
TEST 4: Extracción de Texto (OCR)
══════════════════════════════════════════════════════════════════════

Archivo: 2xpmanzdwefa1.jpg
Prompt: Extract and read all text visible in this image. List it exactly as written.

Extrayendo texto...

✓ Texto extraído:

Sun. 7 8:50 pm
Plus, the fruits of the wild are growing everywhere.
Limus

Detalles:
  Tokens utilizados: 2218
  Modelo: vision


In [36]:
# TEST 5: Análisis de gráfico/chart
print("\n" + "═" * 70)
print("TEST 5: Análisis de Gráfico")
print("═" * 70)

image_path_chart = "image.png"
prompt_chart = """Analyze this chart/graph in detail:
1. What data is being shown?
2. What are the key insights?
3. Which values are highest/lowest?
4. What is the likely use case?"""

print(f"\nArchivo: {image_path_chart}")
print(f"Análisis: Evaluación de datos visuales")
print("\nAnalizando...\n")

try:
    result = analyze_image_from_base64(image_path_chart, prompt_chart)
    analysis = result.choices[0].message.content
    
    print(f"✓ Análisis completado:")
    print(f"\n{analysis}")
    print(f"\nDetalles:")
    print(f"  Tokens utilizados: {result.usage.total_tokens}")
    print(f"  Modelo: {result.model}")
    
except FileNotFoundError as e:
    print(f"✗ Archivo no encontrado: {e}")
    raise
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    raise


══════════════════════════════════════════════════════════════════════
TEST 5: Análisis de Gráfico
══════════════════════════════════════════════════════════════════════

Archivo: image.png
Análisis: Evaluación de datos visuales

Analizando...

✓ Análisis completado:

This bar chart is called Spain's population (1787–2015). Spain's population is measured along a linear scale of range 0 to 50 along the y-axis. On the x-axis, Spain's population is measured along a categorical scale starting with 1787 and ending with 2015. Spain's population from 1787 to 2015 was steadily gaining in numbers. At 25% from 1787, and by 234% from 1787 in 2015.

Detalles:
  Tokens utilizados: 1908
  Modelo: vision


## 4. Part 3: Análisis de Audio

El modelo `Phi-4-multimodal-instruct` también puede procesar audio. Se carga el audio como `InputAudio` en formato base64.

In [30]:
## Audio Analysis Functions using azure.ai.inference

def analyze_audio(audio_file: str, prompt: str):
    """Analiza un archivo de audio usando Phi-4-multimodal-instruct.
    
    Args:
        audio_file: Ruta del archivo de audio (mp3, wav, m4a, ogg)
        prompt: Pregunta/análisis a realizar
    
    Returns:
        Response object con análisis del modelo
    
    Raises:
        FileNotFoundError: Si el archivo no existe
        Exception: Si falla la petición a la API
    """
    if not client:
        raise ValueError("Cliente no configurado")
    
    if not os.path.exists(audio_file):
        raise FileNotFoundError(f"Archivo de audio no encontrado: {audio_file}")
    
    # Determinar audio format
    audio_lower = str(audio_file).lower()
    if audio_lower.endswith('.mp3'):
        audio_format = AudioContentFormat.MP3
    elif audio_lower.endswith('.wav'):
        audio_format = AudioContentFormat.WAV
    elif audio_lower.endswith('.m4a'):
        audio_format = AudioContentFormat.M4A
    elif audio_lower.endswith('.ogg'):
        audio_format = AudioContentFormat.OGG
    else:
        audio_format = AudioContentFormat.MP3
    
    # Load audio using InputAudio
    input_audio = InputAudio.load(
        audio_file=audio_file,
        audio_format=audio_format
    )
    
    response = client.complete(
        messages=[
            SystemMessage("You are an AI assistant for analyzing audio. Transcribe and provide analysis of the audio content."),
            UserMessage(content=[
                TextContentItem(text=prompt),
                AudioContentItem(input_audio=input_audio)
            ]),
        ],
        temperature=1,
        max_tokens=2048,
    )
    
    return response

print("✓ Funciones de análisis de audio definidas (azure.ai.inference SDK)")

✓ Funciones de análisis de audio definidas (azure.ai.inference SDK)


In [34]:
# TEST 3: Análisis de audio
print("\n" + "═" * 70)
print("TEST 3: Análisis de Audio")
print("═" * 70)

# Buscar archivo de audio disponible
audio_files = ["Audio_tarea.mp3"]
audio_found = None

for af in audio_files:
    if os.path.exists(af):
        audio_found = af
        break

if audio_found:
    prompt_audio = "Transcribe and provide a summary of this audio. What is the main topic?"
    
    print(f"\nArchivo de audio: {audio_found}")
    print(f"Prompt: {prompt_audio}")
    print("\nProcesando audio...\n")
    
    try:
        result = analyze_audio(audio_found, prompt_audio)
        analysis = result.choices[0].message.content
        
        print(f"✓ Análisis de audio completado:")
        print(f"\n{analysis}")
        print(f"\nDetalles:")
        print(f"  Tokens utilizados: {result.usage.total_tokens}")
        print(f"  Modelo: {result.model}")
        
    except FileNotFoundError as e:
        print(f"✗ Archivo no encontrado: {e}")
        raise
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
        raise
else:
    print(f"\n⚠️  No se encontró archivo de audio.")
    print(f"   Archivos buscados: {', '.join(audio_files)}")
    print(f"   Para usar esta función, proporciona un archivo de audio en la carpeta actual.")


══════════════════════════════════════════════════════════════════════
TEST 3: Análisis de Audio
══════════════════════════════════════════════════════════════════════

Archivo de audio: Audio_tarea.mp3
Prompt: Transcribe and provide a summary of this audio. What is the main topic?

Procesando audio...

✓ Análisis de audio completado:

Audio caption: A group of people talks about life in Toledo and how to travel in the city.

Detalles:
  Tokens utilizados: 227
  Modelo: speech


## 5. Part 4: Prompts Multimodales (Imagen + Texto + Audio)

Combinación de múltiples modalidades en un solo prompt.

In [32]:
# TEST 6: Prompt multimodal - Imagen + análisis específico
print("\n" + "═" * 70)
print("TEST 6: Prompt Multimodal - Imagen + Análisis Específico")
print("═" * 70)

image_url_product = "https://images.unsplash.com/photo-1560707303-4e980ce876ad?w=400"

prompt_product = """Analyze this image as a product analyst:
1. What is the main product or subject?
2. Estimate the quality and condition
3. Identify any visible branding or text
4. What would be an appropriate use case?
5. Estimate the target audience"""

print(f"\nAnálisis: Evaluación como producto")
print("\nProcesando...\n")

try:
    result = analyze_image_from_url(image_url_product, prompt_product)
    analysis = result.choices[0].message.content
    
    print(f"✓ Análisis multimodal completado:")
    print(f"\n{analysis}")
    print(f"\nDetalles:")
    print(f"  Tokens utilizados: {result.usage.total_tokens}")
    
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    raise


══════════════════════════════════════════════════════════════════════
TEST 6: Prompt Multimodal - Imagen + Análisis Específico
══════════════════════════════════════════════════════════════════════

Análisis: Evaluación como producto

Procesando...

✓ Análisis multimodal completado:

1. The main product or subject is an old, rusty fire hydrant.
2. The quality and condition of the fire hydrant are poor, as it is old and rusty.
3. There is no visible branding or text on the fire hydrant.
4. An appropriate use case for this fire hydrant would be for emergency water supply during a fire or other emergency.
5. The target audience for this fire hydrant would be city officials or utilities responsible for fire safety and emergency services.

Detalles:
  Tokens utilizados: 598


In [37]:
# TEST 7: Solicitar respuesta estructurada en JSON
print("\n" + "═" * 70)
print("TEST 7: Respuesta Estructurada (JSON)")
print("═" * 70)

image_path_json = "image.png"

prompt_json = """Analyze this image and respond ONLY with a valid JSON object (no other text, no markdown):
{
    "title": "brief title",
    "description": "detailed description",
    "main_elements": ["element1", "element2"],
    "colors": ["color1", "color2"],
    "confidence": 0.9
}"""

print(f"\nArchivo: {image_path_json}")
print(f"Solicitando respuesta en formato JSON...\n")

try:
    result = analyze_image_from_base64(image_path_json, prompt_json)
    response_text = result.choices[0].message.content.strip()
    
    # Limpiar markdown si está presente
    if response_text.startswith('```json'):
        response_text = response_text.replace('```json', '').replace('```', '').strip()
    elif response_text.startswith('```'):
        response_text = response_text.replace('```', '').strip()
    
    json_data = json.loads(response_text)
    
    print(f"✓ JSON válido recibido:")
    print(json.dumps(json_data, indent=2, ensure_ascii=False))
    print(f"\nDetalles:")
    print(f"  Tokens utilizados: {result.usage.total_tokens}")
    
except json.JSONDecodeError as e:
    print(f"✗ Error parseando JSON: {e}")
    print(f"Respuesta recibida: {response_text[:200]}...")
    raise
except Exception as e:
    print(f"✗ Error: {type(e).__name__}: {e}")
    raise


══════════════════════════════════════════════════════════════════════
TEST 7: Respuesta Estructurada (JSON)
══════════════════════════════════════════════════════════════════════

Archivo: image.png
Solicitando respuesta en formato JSON...

✓ JSON válido recibido:
{
  "title": "Spain's Population (1787-2015)",
  "description": "The image is a bar chart displaying the population of Spain from 1787 to 2015. The vertical axis represents the population in millions of people, and the horizontal axis represents the years.",
  "main_elements": [
    "bar chart",
    "horizontal axis",
    "vertical axis"
  ],
  "colors": [
    "orange",
    "blue"
  ],
  "confidence": 0.9
}

Detalles:
  Tokens utilizados: 1933
